[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.en/cap06/cap06.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

## 💻 Practical Section with Programming Exercises

The programming exercises (EPs) in this section complement the concepts presented throughout Chapter 6 by implementing algorithms related to industrial inspection and document analysis. The goal is to consolidate the fundamentals studied by reproducing, on a reduced scale, steps of a typical Computer Vision pipeline.

Unlike the previous chapters, whose exercises emphasized operations more directly related to image data, the EPs in this chapter focus on the **intermediate quantities** produced during processing, such as areas, perimeters, circularity, line angles, bubble fill degrees, variance maps, and difference maps. This approach allows each stage of the pipeline to be understood and validated independently, without relying on specialized libraries for image acquisition, marker detection, or code decoding—except for the chapter's closing exercise (EP06_08), which intentionally introduces the use of OpenCV for the segmentation and actual decoding of a QRCode, closing the loop between theoretical concepts and the tools employed in practice.

The exercises follow the same conceptual sequence as the chapter, in increasing order of complexity. Initially, segmentation evaluation metrics are addressed, used to quantify the quality of binary masks. Next, geometric criteria for marker selection, classification of markings on forms, and estimation of document skew through the Hough Transform are studied. In the final part, the exercises explore illumination normalization, defect detection through texture analysis, and the integration of geometric registration and image subtraction in a simplified industrial inspection pipeline.

Each exercise represents an isolated stage of a real Computer Vision system, allowing individual validation of concepts that, in industrial applications, are combined into a single inspection pipeline.

### 🗺️ Difficulty Legend

| Level | Meaning | EPs |
|:---:|---|---|
| 🟢 | Very easy / easy — implementation of a single concept or simple algorithm | EP06_01, EP06_02 |
| 🟡 | Easy–medium — handling multiple cases or using simple statistical criteria | EP06_03, EP06_04 |
| 🟠 | Medium — pointwise matrix processing | EP06_05 |
| 🔴 | Difficult — matrix processing with neighborhood operations (sliding window) | EP06_06 |
| 🟣 | Very difficult — integration of multiple stages of a Computer Vision pipeline | EP06_07 |
| ⚫ | Special — use of specialized library (`cv2`) for geometric segmentation and actual barcode/QRCode decoding | EP06_08 |

> ### ❗ Guidelines for Solving the Programming Exercises
>
> Unless otherwise stated, all exercises use the matrix coordinate convention `[row][column]`, with the origin at $(0,0)$ in the upper-left corner of the image.
>
> When numerical rounding is required, standard rounding to the nearest integer should be used (*round half away from zero*, with `np.floor(img + 0.5)`). Comparisons with thresholds (for example, circularity, variance, intensity difference, or fill degree) should be considered **strict** (`>`), unless the statement explicitly specifies another criterion.
>
> Each exercise was designed to emphasize a specific concept presented in the chapter. It is recommended to initially implement the solution directly and, only after its validation, seek more efficient or more general alternatives.

### 🎯 Objective of this Notebook

This notebook was designed to support the development, validation, and testing of solutions for the **Programming Exercises (PEs)** in an interactive environment, such as Google Colab or Jupyter Notebook. After verifying that the implementation works with the provided test cases, the code can be submitted to Moodle for official evaluation.

#### *Download*

Run the following cell to obtain the files `morph.py` and `testsuite.py`, used by the exercises in this chapter.

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Running the Tests

After implementing the solution, run `TestSuite("EP06_01.extension").run()` in a new cell, replacing `extension` with the language used (`.py`, `.java`, `.c`, `.cpp`, `.js`, or `.r`). The system automatically retrieves the test cases from the course repository, runs the program, and displays the evaluation result.

In Python, it is also possible to test the solution directly from a *string*, without the need to save the code to a file. To do this, store the program in a variable and use the `run_code` method:

```python
codigo = """
# ... your code here ...
"""

TestSuite("EP06_01").run_code(codigo)
```

### EP06_01 🟢 Segmentation Evaluation by IoU (*Intersection over Union*)

Throughout this chapter, several stages of the *pipeline* produce **binary masks**, such as in document segmentation, *QRCode* localization, and defect detection. To objectively evaluate the quality of these segmentations, it is necessary to compare them with a reference mask (*ground truth*).

One of the most widely used metrics for this purpose is the **IoU** (*Intersection over Union*), defined as the ratio between the intersection area and the union area of two binary masks. The higher the IoU value, the greater the agreement between the segmentation produced by the algorithm and the reference.

#### 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (number of rows) and $C$ (number of columns).
2. **Reference mask:** Read the $L \times C$ binary elements (0 or 1) of the `ref` matrix.
3. **Predicted mask:** Read the $L \times C$ binary elements (0 or 1) of the `pred` matrix.
4. **Intersection:** Count the number of positions $(i,j)$ for which `ref[i][j] = 1` and `pred[i][j] = 1`.
5. **Union:** Count the number of positions $(i,j)$ for which `ref[i][j] = 1` or `pred[i][j] = 1`.
6. **Degenerate case:** If the union equals $0$, set $\mathrm{IoU}=1{,}0$, since both masks are empty.
7. **Calculation:** If the union is greater than zero, compute

$$
\mathrm{IoU}=
\frac{|\mathrm{Intersecao}|}
{|\mathrm{Uniao}|}.
$$

8. **Classification:** Determine the qualitative classification using the IoU value **before** rounding.
9. **Rounding:** Display the IoU with four decimal places.
10. **Output:** Print, in this order, the intersection, union, IoU, and classification.

#### 📌 Computational Restrictions

- If the union equals $0$, the division must not be performed; the IoU must be defined as $1{,}0$.
- The classification ranges use non-strict comparisons ($\geq$).
- The classification must be performed using the IoU value in full precision, before rounding for display.

#### 🧠 Theoretical Foundation

The IoU is defined by

$$
\mathrm{IoU}=
\frac{|R\cap P|}
{|R\cup P|},
$$

where:

- $R$ represents the set of pixels belonging to the reference mask;
- $P$ represents the set of pixels belonging to the predicted mask;
- $|R\cap P|$ corresponds to the number of pixels belonging simultaneously to both masks;
- $|R\cup P|$ corresponds to the number of pixels belonging to at least one of the masks.

| IoU Range | Classification | Interpretation |
|---|---|---|
| $\mathrm{IoU}\geq0{,}90$ | `EXCELENTE` | Very high agreement between the masks. |
| $0{,}70\leq\mathrm{IoU}<0{,}90$ | `BOM` | Small differences between the masks. |
| $0{,}50\leq\mathrm{IoU}<0{,}70$ | `ACEITAVEL` | Partial agreement between the masks. |
| $\mathrm{IoU}<0{,}50$ | `RUIM` | Low agreement between the masks. |

The IoU depends only on the overlap between the masks and is therefore independent of the image size.

#### 📦 Input and Output Specification (VPL)

**Input:**

- Line 1: integer $L$.
- Line 2: integer $C$.
- Next $L$ lines: binary elements (0 or 1) of the `ref` matrix.
- Next $L$ lines: binary elements (0 or 1) of the `pred` matrix.

**Output:**

- Line 1: `Intersecao: X`
- Line 2: `Uniao: Y`
- Line 3: `IoU: Z`
- Line 4: `Classificacao: NOME`

The `IoU` value must be printed with four decimal places.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 2<br>2<br>1 1<br>0 0<br>1 0<br>0 0 | Intersecao: 1<br>Uniao: 2<br>IoU: 0.5000<br>Classificacao: ACEITAVEL | Half of the reference region was correctly segmented. |
| 2<br>2<br>0 0<br>0 0<br>0 0<br>0 0 | Intersecao: 0<br>Uniao: 0<br>IoU: 1.0000<br>Classificacao: EXCELENTE | Both masks are empty; by convention, $\mathrm{IoU}=1{,}0$. |

In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-ep0601" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0601 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0601 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0601 button:hover { background: #e8dfcf; }
  #sim-ep0601 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0601_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0601_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0601_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 12px; }
  .sim-ep0601_px { width: 24px; height: 24px; border: 1px solid #e4dcc8; box-sizing: border-border; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP06_01: IoU (Intersection over Union)</span>
  <span class="sim-ep0601_pill">IoU = |A &cap; B| / |A &cup; B|</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles -->
  <div class="sim-ep0601_panel" style="margin-bottom:14px;">
    <div class="sim-ep0601_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Horizontal Shift (&Delta;x)</label>
          <span id="sim-ep0601_vdx" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dx" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Vertical Shift (&Delta;y)</label>
          <span id="sim-ep0601_vdy" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dy" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Square Side</label>
          <span id="sim-ep0601_vsz" style="font-family:monospace; font-weight:700; color:#26241d;">6</span>
        </div>
        <input id="sim-ep0601_sz" type="range" min="2" max="8" step="1" value="6">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">Move and resize the predicted mask to evaluate alignment.</div>
  </div>

  <!-- Exibição das Máscaras 10x10 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">Reference (A)</div>
      <div id="sim-ep0601_ref" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">Predicted (B)</div>
      <div id="sim-ep0601_pred" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">Overlap (A &cap; B)
      </div>
      <div id="sim-ep0601_mix" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0601_dbg" class="sim-ep0601_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep01(root){
    if (!root || root.dataset.sim06Ep01Init) return;
    root.dataset.sim06Ep01Init = "1";

    var dx = root.querySelector("#sim-ep0601_dx");
    var dy = root.querySelector("#sim-ep0601_dy");
    var sz = root.querySelector("#sim-ep0601_sz");

    var vdx = root.querySelector("#sim-ep0601_vdx");
    var vdy = root.querySelector("#sim-ep0601_vdy");
    var vsz = root.querySelector("#sim-ep0601_vsz");

    var gRef  = root.querySelector("#sim-ep0601_ref");
    var gPred = root.querySelector("#sim-ep0601_pred");
    var gMix  = root.querySelector("#sim-ep0601_mix");

    var dbg = root.querySelector("#sim-ep0601_dbg");

    var N = 10;
    var ref = {x: 2, y: 2, w: 6, h: 6};

    function inside(x, y, r){
      return x >= r.x && x < r.x + r.w && y >= r.y && y < r.y + r.h;
    }

    function pixel(color){
      var d = document.createElement("div");
      d.className = "sim-ep0601_px";
      d.style.background = color;
      return d;
    }

    function classe(i){
      if (i >= 0.90) return "Excelente";
      if (i >= 0.75) return "Muito boa";
      if (i >= 0.50) return "Aceitável";
      return "Ruim";
    }

    function render(){
      vdx.textContent = dx.value;
      vdy.textContent = dy.value;
      vsz.textContent = sz.value;

      gRef.innerHTML  = "";
      gPred.innerHTML = "";
      gMix.innerHTML  = "";

      var pred = {
        x: ref.x + parseInt(dx.value, 10),
        y: ref.y + parseInt(dy.value, 10),
        w: parseInt(sz.value, 10),
        h: parseInt(sz.value, 10)
      };

      var inter = 0;
      var uniao = 0;

      for (var y = 0; y < N; y++){
        for (var x = 0; x < N; x++){
          var r = inside(x, y, ref);
          var p = inside(x, y, pred);

          gRef.appendChild(pixel(r ? "#7fdc92" : "#ffffff"));
          gPred.appendChild(pixel(p ? "#7fbfff" : "#ffffff"));

          if (r && p){
            gMix.appendChild(pixel("#9b59b6"));
            inter++;
          }
          else if (r){
            gMix.appendChild(pixel("#7fdc92"));
            uniao++;
          }
          else if (p){
            gMix.appendChild(pixel("#7fbfff"));
            uniao++;
          }
          else{
            gMix.appendChild(pixel("#ffffff"));
          }

          if (r && p) uniao++;
        }
      }

      var iou = inter / uniao;

      dbg.innerHTML =
        "<b>Interseção</b> = " + inter + " pixels &nbsp;&nbsp;&nbsp;" +
        "<b>União</b> = " + uniao + " pixels<br><br>" +
        "IoU = <b>" + inter + " / " + uniao + " = " + iou.toFixed(4) + "</b><br><br>" +
        "<span style='font-weight:700; color:#04342C;'>" + classe(iou) + "</span>";
    }

    dx.addEventListener('input', render);
    dy.addEventListener('input', render);
    sz.addEventListener('input', render);

    render();
  }

  function tryInitSim06Ep01(){
    var root = document.getElementById('sim-ep0601');
    if (root) initSim06Ep01(root); else setTimeout(tryInitSim06Ep01, 200);
  }
  tryInitSim06Ep01();
})();
</script>
""")

**Figure 6.1:** EP06_01 Simulator: IoU between reference mask and predicted mask


<figure id="fig-06-sim-ep0601">
  <img src="imagens/fig-06-sim-ep0601.png" alt=" EP06_01 Simulator: IoU between reference mask and predicted mask " style="max-width:80%" />
  <figcaption><strong>Figure 6.1:</strong>  EP06_01 Simulator: IoU between reference mask and predicted mask </figcaption>
</figure>

In [ ]:
%%writefile EP06_01.py
# Python code

In [ ]:
TestSuite("EP06_01.py").run()

### EP06_02 🟢 Marker Filtering by Circularity

After image segmentation, it is common for several connected components to be identified. In applications such as document rectification, only a few of these components correspond to the reference markers used for image alignment. A criterion often employed to select these markers is **circularity**, which measures how close a component's shape is to a circle.

In this exercise, each component is described by its area $A$ and its perimeter $P$. The goal is to compute its circularity and decide, based on a provided threshold, whether the component should be accepted or rejected as a marker candidate.

#### 📋 Implementation Guidelines

1. **Quantity:** Read the integer $N$ (number of candidates) and the circularity threshold $C_{\text{limiar}}$ (real number).
2. **Candidate data:** For each of the $N$ candidates, read the area $A$ (integer) and the perimeter $P$ (real number).
3. **Circularity:** Compute $C=\frac{4\pi A}{P^2}$, where:

- $A$ is the component's area;
- $P$ is the component's perimeter;
- $C$ is the circularity.

4. **Degenerate case:** If $P=0$, consider $C=0$ and directly classify the candidate as `REJECTED`.
5. **Classification:** If $C>C_{\text{limiar}}$, classify the candidate as `ACCEPTED`; otherwise, classify it as `REJECTED`.
6. **Rounding:** Display the value of $C$ with four decimal places.
7. **Output:** For each candidate, print the value of $C$ followed by the classification. At the end, print the total number of accepted candidates.

#### 📌 Computational Constraints

- Use the constant $\pi$ from the language's standard library (e.g., `math.pi`), without approximations.
- The comparison must be performed using the full-precision value of $C$, before rounding for display.
- The acceptance criterion is strict ($C>C_{\text{limiar}}$).
- If $P=0$, the division must not be performed.

#### 🧠 Theoretical Background

Circularity is a geometric descriptor defined by $C=\frac{4\pi A}{P^2}$, where:

- $A$ is the component's area;
- $P$ is the component's perimeter;
- $C$ is the circularity.

For a perfect circle, $C=1$. As the shape becomes more elongated or irregular, the perimeter grows faster than the area, reducing the value of $C$.

| Shape | Approximate circularity | Interpretation |
|---|---:|---|
| Circle | $1{,}0000$ | Circular shape. |
| Square | $0{,}7854$ | Approximately compact shape. |
| Elongated or irregular shape | $C\ll1$ | Low circularity. |
| $P=0$ | $0$ (adopted convention) | Degenerate contour. |

Circularity is invariant to translation, rotation, and scaling, and it is widely used to distinguish approximately circular components from other shapes.

#### 📦 Input and Output Specification (VPL)

**Input:**

- Line 1: integer $N$.
- Line 2: real number $C_{\text{limiar}}$.
- Next $N$ lines: area $A$ (integer) and perimeter $P$ (real), separated by a space.

**Output:**

- One line for each candidate, in the format `C ACCEPTED` or `C REJECTED`, with $C$ presented with four decimal places.
- Last line: `Total accepted: X`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 3<br>0.6<br>78 31.4<br>100 40<br>50 60 | 0.9941 ACCEPTED<br>0.7854 ACCEPTED<br>0.1745 REJECTED<br>Total accepted: 2 | Approximately circular candidate, compact shape, and elongated shape. |
| 1<br>0.9<br>10 0 | 0.0000 REJECTED<br>Total accepted: 0 | Null perimeter: degenerate contour. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0602" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0602 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0602 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0602 button:hover { background: #e8dfcf; }
  #sim-ep0602 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0602_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0602_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP06_02: Marker Filter by Circularity</span>
  <span class="sim-ep0602_pill">C = 4&pi;A / P&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0602_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Circularity Threshold (C_threshold): <span id="sim-ep0602_vl" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
    </div>
    
    <input id="sim-ep0602_sl" type="range" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Adjust the threshold and observe which candidates (disks, squares, and irregular shapes) survive the filter.
    </div>
  </div>

  <!-- Cards de Candidatos -->
  <div id="sim-ep0602_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(100px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0602_debug" class="sim-ep0602_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep02(root){
    if (!root || root.dataset.sim06Ep02Init) return;
    root.dataset.sim06Ep02Init = "1";

    var candidatos = [
      {nome: "Disco", A: 78, P: 31.4},
      {nome: "Quadrado", A: 100, P: 40},
      {nome: "Retângulo", A: 60, P: 44},
      {nome: "Rasura", A: 50, P: 60},
      {nome: "Ponto", A: 10, P: 0}
    ];

    var slEl  = root.querySelector('#sim-ep0602_sl');
    var vlEl  = root.querySelector('#sim-ep0602_vl');
    var cards = root.querySelector('#sim-ep0602_cards');
    var dbg   = root.querySelector('#sim-ep0602_debug');

    function render(){
      var th = parseFloat(slEl.value);
      vlEl.textContent = th.toFixed(2);
      cards.innerHTML = '';
      var aceitos = 0;

      candidatos.forEach(function(c){
        var C = (c.P === 0) ? 0 : (4 * Math.PI * c.A) / (c.P * c.P);
        var ok = c.P !== 0 && C > th;
        if (ok) aceitos++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        
        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + c.nome + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px; opacity:0.8;">A = ' + c.A + '<br>P = ' + c.P + '</div>' +
          '<div style="font-family:monospace; font-weight:700; margin-bottom:4px;">C = ' + C.toFixed(4) + '</div>' +
          '<div style="font-weight:700; font-size:10px; letter-spacing:0.04em;">' + (ok ? 'ACEITO' : 'REJEITADO') + '</div>';
        
        cards.appendChild(div);
      });

      dbg.textContent = 'C_threshold = ' + th.toFixed(2) + '  |  Candidatos aceitos: ' + aceitos + ' / ' + candidatos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep02(){
    var root = document.getElementById('sim-ep0602');
    if (root) initSim06Ep02(root); else setTimeout(tryInitSim06Ep02, 200);
  }
  tryInitSim06Ep02();
})();
</script>
""")

**Figure 6.2:** Simulator EP06_02: Marker Filter by Circularity


<figure id="fig-06-sim-ep0602">
  <img src="imagens/fig-06-sim-ep0602.png" alt=" Simulator EP06_02: Marker Filter by Circularity " style="max-width:80%" />
  <figcaption><strong>Figure 6.2:</strong>  Simulator EP06_02: Marker Filter by Circularity </figcaption>
</figure>

In [ ]:
%%writefile EP06_02.py
# Python code

In [ ]:
TestSuite("EP06_02.py").run()

### EP06_03 🟡 Classification of Markings on Answer Sheets (OMR)

After sheet rectification and segmentation of answer frames, MCTest estimates, for each bubble, a **fill degree**, represented by a value between $0$ and $100$. Based on these values, the system must automatically determine the marked alternative, also identifying blank questions and cases of multiple markings.

In this exercise, you will implement this decision stage of the OMR *pipeline*. The classification depends on a fill threshold: small variations in this value can alter the result of automatic reading.

#### 📋 Implementation Guidelines

1. **Parameters:** Read integers $Q$ (number of questions) and $K$ (number of alternatives per question, with $2 \le K \le 26$) and the fill threshold $\mathrm{Th}$ (a real number between $0$ and $100$).
2. **Fill degrees:** For each of the $Q$ questions, read the $K$ real values corresponding to alternatives `A`, `B`, `C`, ..., in input order.
3. **Counting markings:** For each question, count how many alternatives have a fill degree **strictly greater** than $\mathrm{Th}$.
4. **Classification:**
   - If no alternative exceeds $\mathrm{Th}$, classify the question as `BRANCO`.
   - If exactly one alternative exceeds $\mathrm{Th}$, print the corresponding letter (`A`, `B`, `C`, ...).
   - If two or more alternatives exceed $\mathrm{Th}$, classify the question as `DUPLA_MARCACAO`.
5. **Output per question:** Print, in reading order, the classification of each question.
6. **Totals:** Finally, print the number of questions `OK` (single marking), `BRANCO`, and `DUPLA_MARCACAO`.

#### 📌 Computational Constraints

* **Strict comparison:** only values greater than $\mathrm{Th}$ are considered valid markings; values exactly equal to the threshold must not be counted.
* **Alternative letters:** index $0$ corresponds to alternative `A`, index $1$ to alternative `B`, and so on.
* **Multiple markings:** whenever two or more alternatives exceed the threshold, the classification must be `DUPLA_MARCACAO`, regardless of their respective fill degrees.

#### 🧠 Theoretical Foundation

| Situation | Classification | Interpretation |
|---|---|---|
| Exactly one alternative above the threshold | Letter of the alternative | Valid answer |
| No alternative above the threshold | `BRANCO` | Unanswered question |
| Two or more alternatives above the threshold | `DUPLA_MARCACAO` | Ambiguous answer |

The fill threshold controls the sensitivity of the algorithm. Very low values tend to increase the number of `DUPLA_MARCACAO`, while very high values may increase the number of questions classified as `BRANCO`.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $Q$.
* Line 2: Integer $K$.
* Line 3: Real number $\mathrm{Th}$.
* Next $Q$ lines: $K$ real numbers, corresponding to the fill degrees of the alternatives.
* Line 1: Integer $Q$ and $K$.

**Output:**

* $Q$ lines, each containing the classification of the respective question.
* Final line: `OK: x  BRANCO: y  DUPLA_MARCACAO: z`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 3<br>4<br>50<br>10 85 5 12<br>20 15 18 22<br>90 88 10 5 | B<br>BRANCO<br>DUPLA_MARCACAO<br>OK: 1  BRANCO: 1  DUPLA_MARCACAO: 1 | In the first question only `B` exceeds the threshold; in the second, no alternative exceeds it; in the third, `A` and `B` exceed the threshold. |
| 1<br>2<br>50.0<br>50 50 | BRANCO<br>OK: 0  BRANCO: 1  DUPLA_MARCACAO: 0 | Values equal to the threshold are not considered valid markings. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0603" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0603 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0603 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0603 button:hover { background: #e8dfcf; }
  #sim-ep0603 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0603_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0603_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP06_03: OMR Mark Classification</span>
  <span class="sim-ep0603_pill">4 Alternatives</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Fill Threshold (Th): <span id="sim-ep0603_vth" style="font-family:monospace; color:#26241d;">50</span>%
      </label>
    </div>
    
    <input id="sim-ep0603_th" type="range" min="0" max="100" step="1" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Adjust the fill level of each bubble (A&ndash;D) and the threshold to observe the resulting classification.
    </div>
  </div>

  <!-- Sliders das Bolhas (A-D) -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div id="sim-ep0603_bubbles" style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0603_debug" class="sim-ep0603_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep03(root){
    if (!root || root.dataset.sim06Ep03Init) return;
    root.dataset.sim06Ep03Init = "1";

    var letras = ['A', 'B', 'C', 'D'];
    var valores = [10, 85, 5, 12];
    var thEl  = root.querySelector('#sim-ep0603_th');
    var vthEl = root.querySelector('#sim-ep0603_vth');
    var box   = root.querySelector('#sim-ep0603_bubbles');
    var dbg   = root.querySelector('#sim-ep0603_debug');

    box.innerHTML = '';
    var sliders = [];

    letras.forEach(function(L, i){
      var col = document.createElement('div');
      col.style.cssText = 'text-align:center; background:#fafaf7; border:1px solid #e9e3d3; padding:10px; border-radius:8px;';
      col.innerHTML = '<div style="font-weight:700; font-size:12px; color:#5e5a4a; margin-bottom:6px;">' + L + '</div>' +
        '<input type="range" min="0" max="100" step="1" value="' + valores[i] + '" id="sim-ep0603_b' + i + '">' +
        '<div id="sim-ep0603_v' + i + '" style="font-family:monospace; font-weight:700; font-size:11px; color:#26241d; margin-top:6px;">' + valores[i] + '%</div>';
      box.appendChild(col);
      sliders.push(col.querySelector('#sim-ep0603_b' + i));
    });

    function render(){
      var th = parseFloat(thEl.value);
      vthEl.textContent = th.toFixed(0);
      var marcadas = [];

      sliders.forEach(function(s, i){
        var v = parseFloat(s.value);
        root.querySelector('#sim-ep0603_v' + i).textContent = v.toFixed(0) + '%';
        if (v > th) marcadas.push(letras[i]);
      });

      var resultado;
      if (marcadas.length === 0) {
        resultado = 'BRANCO';
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#8a8371';
      } else if (marcadas.length === 1) {
        resultado = 'RESPOSTA: ' + marcadas[0];
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        resultado = 'DUPLA_MARCACAO (' + marcadas.join(', ') + ')';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'Question classification: ' + resultado;
    }

    sliders.forEach(function(s){ s.addEventListener('input', render); });
    thEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep03(){
    var root = document.getElementById('sim-ep0603');
    if (root) initSim06Ep03(root); else setTimeout(tryInitSim06Ep03, 200);
  }
  tryInitSim06Ep03();
})();
</script>
""")

**Figure 6.3:** EP06_03 Simulator: OMR Mark Classification


<figure id="fig-06-sim-ep0603">
  <img src="imagens/fig-06-sim-ep0603.png" alt=" EP06_03 Simulator: OMR Mark Classification " style="max-width:80%" />
  <figcaption><strong>Figure 6.3:</strong>  EP06_03 Simulator: OMR Mark Classification </figcaption>
</figure>

In [ ]:
%%writefile EP06_03.py
# Python code

In [ ]:
TestSuite("EP06_03.py").run()

### EP06_04 🟡 Median Angular Slope Estimator (*Deskew*)

After edge detection and the application of the Hough Transform, a set of candidate lines for the predominant orientation of the document is obtained. Each line provides an estimate of the skew angle, calculated by

$$
\text{angle} = \operatorname{rad2deg}(\theta) - 90.
$$

However, not all lines correspond to document lines: some result from noise, shadows, or other image elements. In this exercise, you will implement the robust estimation step of the skew angle, filtering plausible values and computing their median.

#### 📋 Implementation Guidelines

1. **Quantity:** Read the integer $M$, corresponding to the number of estimated angles.
2. **Angles:** Read the $M$ real values, in degrees.
3. **Filtering:** Keep only the angles that **strictly** satisfy $-45 < \text{angle} < 45$.
4. **No candidates:** If no angle remains after filtering, print exactly `SEM_CORRECAO`.
5. **Median:** If valid angles exist:
   - if the quantity is odd, the median is the central element of the ordered sequence;
   - if it is even, the median is the arithmetic mean of the two central elements.
6. **Output:** Print the median rounded to two decimal places (standard rounding, *round half away from zero*, with `np.floor(img + 0.5)`).

#### 📌 Computational Constraints

* **Open interval:** angles equal to $-45$ or $45$ must not be considered.
* **Precision:** compute the median using the original values; rounding must be performed only at the output.
* **Empty case:** if there are no valid angles, no median must be computed.

#### 🧠 Theoretical Background

| Situation | Result |
|---|---|
| Most angles concentrated around the true skew | The median approximates the document orientation. |
| Few discrepant angles (*outliers*) | The median is little influenced by these values. |
| Angles outside the interval $(-45^\circ,45^\circ)$ | They are discarded before the computation. |
| No valid angle | No correction is applied (`SEM_CORRECAO`). |

The median is used because it is more robust than the mean in the presence of a few discrepant values, producing a more stable estimate of the predominant document skew.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $M$.
* Line 2: $M$ real numbers, corresponding to the angles in degrees.

**Output:**

* A single line containing the estimated angle, with two decimal places, or the word `SEM_CORRECAO` if no angle is valid.

#### 📌 Examples

| Input | Output | Remark |
|---|---|---|
| 5<br>-50 -10.5 2.3 2.3 47 | 2.30 | Only angles in the interval $(-45,45)$ are considered; the median is $2.3$. |
| 4<br>-46 50 45 -45 | SEM_CORRECAO | No angle belongs to the open interval $(-45,45)$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0604" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0604 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0604 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0604 button:hover { background: #e8dfcf; }
  #sim-ep0604 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0604_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0604_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP06_04: Angular Median Slope Estimator (Deskew)</span>
  <span class="sim-ep0604_pill">median(-45&deg; &lt; &theta; &lt; 45&deg;)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Extra Noise Angle (&theta;_noise): <span id="sim-ep0604_vl" style="font-family:monospace; color:#26241d;">47</span>&deg;
      </label>
    </div>
    
    <input id="sim-ep0604_sl" type="range" min="-80" max="80" step="1" value="47">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Drag the extra noise angle inside or outside the interval [-45&deg;, +45&deg;] and see how the median remains stable.
    </div>
  </div>

  <!-- Exibição dos Ângulos Amostrados -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; text-align:center; letter-spacing:0.04em;">
      Angle Samples (Green = Within Range, Red = Discarded Noise)
    </div>
    <div id="sim-ep0604_pts" style="display:flex; gap:8px; flex-wrap:wrap; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0604_debug" class="sim-ep0604_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep04(root){
    if (!root || root.dataset.sim06Ep04Init) return;
    root.dataset.sim06Ep04Init = "1";

    var base = [-10.5, 2.3, 2.3];
    var slEl  = root.querySelector('#sim-ep0604_sl');
    var vlEl  = root.querySelector('#sim-ep0604_vl');
    var ptsEl = root.querySelector('#sim-ep0604_pts');
    var dbg   = root.querySelector('#sim-ep0604_debug');

    function median(arr){
      var a = arr.slice().sort(function(x, y){ return x - y; });
      var n = a.length;
      if (n === 0) return null;
      var mid = Math.floor(n / 2);
      return (n % 2 === 1) ? a[mid] : (a[mid - 1] + a[mid]) / 2;
    }

    function render(){
      var extra = parseFloat(slEl.value);
      vlEl.textContent = extra;
      var todos = base.concat([extra, -50]);
      var validos = todos.filter(function(a){ return a > -45 && a < 45; });
      
      ptsEl.innerHTML = '';
      todos.forEach(function(a){
        var ok = a > -45 && a < 45;
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 12px; border-radius:8px; font-family:monospace; font-size:12px; font-weight:700; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        div.textContent = a + '°';
        ptsEl.appendChild(div);
      });

      var med = median(validos);

      if (med === null) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
        dbg.textContent = 'Válidos: []  |  Mediana estimada: SEM_CORRECAO';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
        dbg.textContent = 'Valid: [' + validos.join(', ') + ']  |  Mediana estimada: ' + med.toFixed(2) + '°';
      }
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep04(){
    var root = document.getElementById('sim-ep0604');
    if (root) initSim06Ep04(root); else setTimeout(tryInitSim06Ep04, 200);
  }
  tryInitSim06Ep04();
})();
</script>
""")

**Figure 6.4:** EP06_04 Simulator: Slope Estimator by Angular Median


<figure id="fig-06-sim-ep0604">
  <img src="imagens/fig-06-sim-ep0604.png" alt=" EP06_04 Simulator: Slope Estimator by Angular Median " style="max-width:80%" />
  <figcaption><strong>Figure 6.4:</strong>  EP06_04 Simulator: Slope Estimator by Angular Median </figcaption>
</figure>

In [ ]:
%%writefile EP06_04.py
# Python code

In [ ]:
TestSuite("EP06_04.py").run()

### EP06_05 🟠 Background Normalization by Division (Illumination Correction)

A form was photographed under non-uniform illumination, causing one side of the page to appear lighter than the other. Under these conditions, global thresholding by Otsu can produce unsatisfactory results, since a single threshold does not properly separate text and background across the entire image. The solution presented in the chapter consists of **normalizing the background** by dividing the original image by a heavily smoothed version of itself, which represents the low-frequency illumination.

In this exercise, the original image and the smoothed background (equivalent to the result of a `cv2.GaussianBlur` with high $\sigma$) are already provided. Your task is to implement the normalization step that produces the corrected image.

#### 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Original image:** Read the $L \times C$ integer values of the matrix `img` (intensities between 0 and 255).
3. **Estimated background:** Read the $L \times C$ integer values of the matrix `bg` (intensities between 0 and 255, always strictly greater than zero).
4. **Normalization:** For each position $(i,j)$, compute
$$
\text{value}(i,j)=
\frac{\text{img}(i,j)}{\text{bg}(i,j)}\times255.
$$
5. **Rounding:** Round the result to the nearest integer (*round half away from zero*, using `np.floor(img + 0.5)`).
6. **Saturation:** Clip the obtained value to the interval $[0,255]$.
7. **Output:** Print the resulting matrix `img_norm`.

#### 📌 Computational Constraints

* **Division by zero:** the input guarantees $\text{bg}(i,j)>0$ at all positions.
* **Order of operations:** first round, then apply saturation.
* **Independent processing:** each pixel must be normalized individually, without using information from neighboring pixels.

#### 🧠 Theoretical Foundation

| Situation | Effect of normalization |
|---|---|
| $\text{img}(i,j)=\text{bg}(i,j)$ | Result equal to $255$, corresponding to the normalized background. |
| $\text{img}(i,j)<\text{bg}(i,j)$ | Result less than $255$, preserving darker regions, such as text. |
| $\text{img}(i,j)>\text{bg}(i,j)$ | Result greater than $255$, subsequently saturated. |
| Background with non-uniform illumination | The division reduces slow illumination variations, making the image more homogeneous. |

Dividing by the estimated background reduces the effects of non-uniform illumination and preserves the contrast between foreground and background, facilitating subsequent segmentation steps.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Next $L$ lines: elements of the matrix `img`.
* Next $L$ lines: elements of the matrix `bg`.

**Output:**

* Matrix `img_norm`, with $L$ rows and $C$ columns, containing integer values separated by spaces.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 2<br>2<br>60 120<br>180 40<br>100 100<br>200 80 | 153 255<br>230 128 | Values greater than $255$ must be saturated; $180/200\times255=229.5$ results in $230$ after rounding. |
| 1<br>3<br>30 60 90<br>60 60 60 | 128 255 255 | Only the first value remains below $255$ after normalization. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0605" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0605 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0605 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0605 button:hover { background: #e8dfcf; }
  #sim-ep0605 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0605_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0605_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05_ep05_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 9px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP06_05: Background Normalization by Division</span>
  <span class="sim-ep0605_pill">(img / bg) &times; 255</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0605_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Left Background Intensity (bg_esq): <span id="sim-ep0605_vl" style="font-family:monospace; color:#26241d;">100</span>
      </label>
    </div>
    
    <input id="sim-ep0605_sl" type="range" min="40" max="220" step="5" value="100">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Adjust the background gradient (left &rarr; right) and observe how division cancels the illumination variation.
    </div>
  </div>

  <!-- Exibição das Matrizes 1x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img (Original)
      </div>
      <div id="sim-ep0605_g_img" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        bg (Smoothed Background)
      </div>
      <div id="sim-ep0605_g_bg" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img_norm (Output)
      </div>
      <div id="sim-ep0605_g_out" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0605_debug" class="sim-ep0605_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep05(root){
    if (!root || root.dataset.sim06Ep05Init) return;
    root.dataset.sim06Ep05Init = "1";

    var linha_img = [90, 90, 90, 90];
    var slEl = root.querySelector('#sim-ep0605_sl');
    var vlEl = root.querySelector('#sim-ep0605_vl');
    var gImg = root.querySelector('#sim-ep0605_g_img');
    var gBg  = root.querySelector('#sim-ep0605_g_bg');
    var gOut = root.querySelector('#sim-ep0605_g_out');
    var dbg  = root.querySelector('#sim-ep0605_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function cellStyle(v){
      var g = Math.max(0, Math.min(255, v));
      return 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var bgEsq = parseInt(slEl.value, 10);
      vlEl.textContent = bgEsq;

      // Gradiente linear de bgEsq até 200 na direita, 4 colunas
      var bg = [];
      for (var j = 0; j < 4; j++){
        bg.push(Math.round(bgEsq + (200 - bgEsq) * j / 3));
      }

      gImg.innerHTML = '';
      gBg.innerHTML  = '';
      gOut.innerHTML = '';
      
      var out = [];
      for (var j = 0; j < 4; j++){
        var v = (linha_img[j] / bg[j]) * 255;
        var r = roundHalfAway(v);
        var sat = Math.max(0, Math.min(255, r));
        out.push(sat);

        var ci = document.createElement('div');
        ci.className = 'sim05_ep05_cell';
        ci.style.cssText = cellStyle(linha_img[j]);
        ci.textContent = linha_img[j];
        gImg.appendChild(ci);

        var cb = document.createElement('div');
        cb.className = 'sim05_ep05_cell';
        cb.style.cssText = cellStyle(bg[j]);
        cb.textContent = bg[j];
        gBg.appendChild(cb);

        var co = document.createElement('div');
        co.className = 'sim05_ep05_cell';
        co.style.cssText = cellStyle(sat);
        co.textContent = sat;
        gOut.appendChild(co);
      }

      dbg.textContent = 'bg = [' + bg.join(', ') + ']  |  img_norm = [' + out.join(', ') + ']';
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep05(){
    var root = document.getElementById('sim-ep0605');
    if (root) initSim06Ep05(root); else setTimeout(tryInitSim06Ep05, 200);
  }
  tryInitSim06Ep05();
})();
</script>
""")

**Figure 6.5:** EP06_05 Simulator: Background Normalization by Division


<figure id="fig-06-sim-ep0605">
  <img src="imagens/fig-06-sim-ep0605.png" alt=" EP06_05 Simulator: Background Normalization by Division " style="max-width:80%" />
  <figcaption><strong>Figure 6.5:</strong>  EP06_05 Simulator: Background Normalization by Division </figcaption>
</figure>

In [ ]:
%%writefile EP06_05.py
# Python code

In [ ]:
TestSuite("EP06_05.py").run()

### EP06_06 🔴 Local Variance Map for Texture Detection

A fabric factory needs to inspect rolls of cloth in real time, without having a reference image available — each roll presents small natural variations. In this situation, the strategy presented in the chapter consists of analyzing the **local homogeneity of the texture**: uniform regions exhibit low intensity variance in small neighborhoods, while scratches, stains, and manufacturing defects produce local increases in this variance.

In this exercise, you will implement the core of this method, calculating the local variance in a sliding window and generating a binary mask that identifies regions whose variance exceeds a threshold.

#### 📋 Implementation Guidelines

1. **Dimensions and parameters:** Read the integers $L$, $C$, $k$ (window size, always odd) and $T$ (variance threshold).
2. **Image:** Read the $L \times C$ integer values of the texture matrix (intensities between 0 and 255).
3. **Border handling:** When the window extends beyond the image boundaries, use **border replication**, i.e., repeat the value of the nearest valid pixel.
4. **Local mean:** For each position $(i,j)$, compute
$$
\mu(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{window}}
\text{texture}(p,q).
$$
5. **Local variance:** Compute the population variance of the window,
$$
\sigma^2(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{window}}
\left(\text{texture}(p,q)-\mu(i,j)\right)^2,
$$
or, equivalently,
$$
\sigma^2(i,j)=\overline{x^2}-\mu(i,j)^2,
$$
where $\overline{x^2}$ represents the mean of the squared intensities.

6. **Rounding:** Round the variance to the nearest integer (*round half away from zero*, using `np.floor(res_norm + 0.5)`).

7. **Thresholding:** Set $\text{mask}(i,j)=1$ if the rounded variance is **strictly greater** than $T$; otherwise, set $\text{mask}(i,j)=0$.

8. **Output:** Print the resulting binary mask.

#### 📌 Computational Constraints

* **Border replication:** use the value of the nearest valid pixel whenever the window extends beyond the image boundaries.
* **Population variance:** use denominator $k^2$, never $k^2-1$.
* **Strict comparison:** the mask must be computed using the condition $\sigma^2_{\text{rounded}}>T$.
* **Odd window:** the value of $k$ is always odd, ensuring a central pixel.

#### 🧠 Theoretical Foundation

| Situation | Local variance | Interpretation |
|---|---|---|
| Uniform region | Low | Similar intensities in the neighborhood. |
| Region containing a defect | High | The presence of distinct intensities increases the dispersion of values. |
| Small window | Greater sensitivity to details and noise | Detects localized changes. |
| Large window | Smoother response | Highlights larger defects, but reduces the precision of their localization. |

Local variance measures the dispersion of intensities in a neighborhood. Homogeneous regions exhibit low variance, while texture changes increase this measure, allowing the identification of potential defects through simple thresholding.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $k$ (odd).
* Line 4: Integer $T$.
* Next $L$ lines: integer elements of the texture matrix.

**Output:**

* Binary mask (values 0 or 1), with $L$ rows and $C$ columns.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 3<br>3<br>3<br>50<br>10 10 10<br>10 10 10<br>10 90 10 | 0 0 0<br>1 1 1<br>1 1 1 | The defect increases the variance in all windows that contain it. |
| 2<br>2<br>3<br>5<br>100 100<br>100 100 | 0 0<br>0 0 | The texture is uniform; the variance is zero throughout the image. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0606" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0606 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0606 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0606 button:hover { background: #e8dfcf; }
  #sim-ep0606 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0606_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0606_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0606_cell { width: 44px; height: 44px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP06_06: Local Variance (Texture Detection)</span>
  <span class="sim-ep0606_pill">&sigma;&sup2; = m&eacute;dia(x&sup2;) &minus; m&eacute;dia(x)&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0606_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Defect Intensity (Center Position): <span id="sim-ep0606_vl_def" style="font-family:monospace; color:#26241d;">90</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_def" type="range" min="10" max="255" step="5" value="90">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Threshold (T): <span id="sim-ep0606_vl_t" style="font-family:monospace; color:#26241d;">50</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_t" type="range" min="0" max="2000" step="10" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Adjust the defect value and the threshold T; observe how the 3&times;3 window spreads detection across the neighborhood.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Texture (3&times;3)
      </div>
      <div id="sim-ep0606_g_tex" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Defect Mask
      </div>
      <div id="sim-ep0606_g_mask" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0606_debug" class="sim-ep0606_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep06(root){
    if (!root || root.dataset.sim06Ep06Init) return;
    root.dataset.sim06Ep06Init = "1";

    var slDef = root.querySelector('#sim-ep0606_sl_def');
    var vlDef = root.querySelector('#sim-ep0606_vl_def');
    var slT   = root.querySelector('#sim-ep0606_sl_t');
    var vlT   = root.querySelector('#sim-ep0606_vl_t');
    var gTex  = root.querySelector('#sim-ep0606_g_tex');
    var gMask = root.querySelector('#sim-ep0606_g_mask');
    var dbg   = root.querySelector('#sim-ep0606_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function clampIdx(v, n){
      return Math.max(0, Math.min(n - 1, v));
    }

    function render(){
      var defeito = parseInt(slDef.value, 10);
      var T       = parseInt(slT.value, 10);
      vlDef.textContent = defeito;
      vlT.textContent   = T;

      var N = 3;
      var tex = [[10, 10, 10], [10, defeito, 10], [10, 10, 10]];

      gTex.innerHTML  = '';
      gMask.innerHTML = '';
      
      var mask = [];
      for (var i = 0; i < N; i++){
        var row = [];
        for (var j = 0; j < N; j++){
          var vals = [];
          for (var di = -1; di <= 1; di++){
            for (var dj = -1; dj <= 1; dj++){
              var pi = clampIdx(i + di, N);
              var pj = clampIdx(j + dj, N);
              vals.push(tex[pi][pj]);
            }
          }
          var mean = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
          var meanSq = vals.reduce(function(a, b){ return a + b * b; }, 0) / vals.length;
          var varr = meanSq - mean * mean;
          var varRound = roundHalfAway(varr);
          row.push(varRound > T ? 1 : 0);
        }
        mask.push(row);
      }

      var total = 0;
      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var g = tex[i][j];
          var ct = document.createElement('div');
          ct.className = 'sim-ep0606_cell';
          ct.style.cssText = 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
          ct.textContent = g;
          gTex.appendChild(ct);

          var m = mask[i][j];
          if (m) total++;

          var cm = document.createElement('div');
          cm.className = 'sim-ep0606_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'defect = ' + defeito + '  |  T = ' + T + '  |  Pixels marcados: ' + total + ' / 9';
    }

    slDef.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep06(){
    var root = document.getElementById('sim-ep0606');
    if (root) initSim06Ep06(root); else setTimeout(tryInitSim06Ep06, 200);
  }
  tryInitSim06Ep06();
})();
</script>
""")

**Figure 6.6:** EP06_06 Simulator: Local Variance Map for Texture Detection


<figure id="fig-06-sim-ep0606">
  <img src="imagens/fig-06-sim-ep0606.png" alt=" EP06_06 Simulator: Local Variance Map for Texture Detection " style="max-width:80%" />
  <figcaption><strong>Figure 6.6:</strong>  EP06_06 Simulator: Local Variance Map for Texture Detection </figcaption>
</figure>

In [ ]:
%%writefile EP06_06.py
# Python code

In [ ]:
TestSuite("EP06_06.py").run()

### EP06_07 🟣 Industrial Inspection Pipeline: Registration by Translation and Subtraction

On a production line, a fixed camera photographs each part as it passes on the conveyor belt, comparing it to a defect-free reference image. The problem: small vibrations in the conveyor shift the part relative to the reference position at each capture. If image subtraction is applied directly, without correction, the displacement alone already generates enormous differences — **false positives** that mask the real defects.

This is the most comprehensive exercise in the chapter: you must **first register** (geometrically align) the captured image using a known displacement $(dx, dy)$, provided by a position sensor on the conveyor, and **only then apply subtraction** with thresholding, exactly as described in the industrial inspection section.

#### 📋 Implementation Guidelines

1. **Dimensions and parameters:** Read $L$, $C$ (image dimensions), the known integer displacement $dx, dy$ (which may be negative), and the detection threshold $T$ (integer).
2. **Images:** Read the reference matrix (`ref`, $L\times C$, defect-free) and the captured matrix (`cap`, $L\times C$, possibly shifted and with defects).
3. **Registration by translation:** Construct the aligned image `alin` by applying the received displacement $(dx,dy)$:
$$
\text{alin}(i,j) = \begin{cases} \text{cap}(i+dy,\; j+dx), & \text{if } (i+dy,\ j+dx) \in [0,L)\times[0,C) \\ 0, & \text{otherwise} \end{cases}
$$
4. **Border filling:** Positions that "leave" the captured image after the displacement are assigned the value **0** (*zero-padding* — outside the camera's field of view; **note that this exercise uses zero, unlike the border replication of EP06_06**).
5. **Absolute difference:** Compute, pixel by pixel,
$$
\text{diff}(i,j) = |\text{ref}(i,j) - \text{alin}(i,j)|
$$
6. **Thresholding:** Define $\text{mask}(i,j) = 1$ if $\text{diff}(i,j) > T$; otherwise, $\text{mask}(i,j) = 0$.
7. **Output:** In this order — (a) the matrix `alin` ($L\times C$); (b) the defect mask ($L\times C$); (c) a final line with the total number of pixels classified as defective.

#### 📌 Computational Constraints

* ***Zero-padding*, not replication:** positions outside the bounds of the captured image, after the displacement, are exactly 0 — this is the point that most differentiates this exercise from EP06_06.
* **Strict comparison:** $\text{diff}(i,j) > T$.
* **Sign of $(dx,dy)$:** the displacement may be positive or negative; the formula in step 3 must be applied literally, without inverting the signs.
* **All values are integers:** there is no rounding at this stage.

#### 🧠 Theoretical Foundation

| Omitted step | Consequence |
|---|---|
| Skipping geometric registration | The entire image border (introduced by the displacement) is marked as "defect" — systematic false positive |
| Registration with incorrect $(dx,dy)$ | Part and reference remain misaligned; subtraction detects shifted contours, not real defects |
| Threshold $T$ too low | Capture noise (variations of 1–2 gray levels) is mistaken for defects |
| Threshold $T$ too high | Subtle defects go undetected |

Geometric registration and subtraction are complementary steps: the former ensures that both images represent exactly the same scene in the same spatial reference frame; the latter isolates what actually changed between them — ideally, only the defects.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Two integers $dx$ and $dy$, separated by a space.
* Line 4: Integer $T$.
* Next $L$ lines: integer elements of the `ref` matrix.
* Next $L$ lines: integer elements of the `cap` matrix.

**Output:**

* $L$ lines with the `alin` matrix.
* $L$ lines with the defect mask (0/1).
* Last line: `Total de pixels defeituosos: X`.

#### 📌 Examples

| Input | Output | Remark |
|---|---|---|
| 3<br>3<br>1 0<br>30<br>50 50 50<br>50 50 50<br>50 50 50<br>0 50 50<br>0 50 90<br>0 50 50 | 50 50 0<br>50 90 0<br>50 50 0<br>0 0 1<br>0 1 1<br>0 0 1<br>Total de pixels defeituosos: 4 | $dx=1$ shifts the reading one column to the right; the last column of `alin` has no correspondence <br> (becomes 0) and is systematically marked; the real defect (90) is also detected. |
| 2<br>2<br>0 0<br>20<br>10 10<br>10 10<br>10 10<br>10 60 | 10 10<br>10 60<br>0 0<br>0 1<br>Total de pixels defeituosos: 1 | No displacement ($dx=dy=0$): `alin` is identical to `cap`; only the real defect (60) is detected. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0607" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0607 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0607 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0607 button:hover { background: #e8dfcf; }
  #sim-ep0607 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0607_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0607_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0607_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 10px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP06_07: Translation Registration + Subtraction</span>
  <span class="sim-ep0607_pill">|ref &minus; align(dx,dy)| &gt; T</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0607_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Horizontal Shift (dx): <span id="sim-ep0607_vl_dx" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_dx" type="range" min="-2" max="2" step="1" value="1">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Threshold (T): <span id="sim-ep0607_vl_t" style="font-family:monospace; color:#26241d;">30</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_t" type="range" min="0" max="100" step="5" value="30">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Adjust the conveyor shift (dx) and threshold T. Watch how the "ghost" edge disappears when dx = 0.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        ref
      </div>
      <div id="sim-ep0607_g_ref" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        align (registered)
      </div>
      <div id="sim-ep0607_g_alin" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        mask
      </div>
      <div id="sim-ep0607_g_mask" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0607_debug" class="sim-ep0607_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep07(root){
    if (!root || root.dataset.sim06Ep07Init) return;
    root.dataset.sim06Ep07Init = "1";

    var N = 3;
    var ref = [[50, 50, 50], [50, 50, 50], [50, 50, 50]];
    // cap representa a peça deslocada 1 px à direita (col 0 = 0) mais um defeito em (1,2)
    var cap = [[0, 50, 50], [0, 50, 90], [0, 50, 50]];

    var slDx  = root.querySelector('#sim-ep0607_sl_dx');
    var vlDx  = root.querySelector('#sim-ep0607_vl_dx');
    var slT   = root.querySelector('#sim-ep0607_sl_t');
    var vlT   = root.querySelector('#sim-ep0607_vl_t');
    var gRef  = root.querySelector('#sim-ep0607_g_ref');
    var gAlin = root.querySelector('#sim-ep0607_g_alin');
    var gMask = root.querySelector('#sim-ep0607_g_mask');
    var dbg   = root.querySelector('#sim-ep0607_debug');

    function cellStyle(g){
      var v = Math.max(0, Math.min(255, g));
      return 'background:rgb(' + v + ',' + v + ',' + v + '); color:' + (v > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var dx = parseInt(slDx.value, 10);
      var T  = parseInt(slT.value, 10);
      vlDx.textContent = dx;
      vlT.textContent  = T;

      gRef.innerHTML  = '';
      gAlin.innerHTML = '';
      gMask.innerHTML = '';

      var alin = [], mask = [], total = 0;

      for (var i = 0; i < N; i++){
        var rowA = [], rowM = [];
        for (var j = 0; j < N; j++){
          var pj = j + dx;
          var v = (pj >= 0 && pj < N) ? cap[i][pj] : 0;
          rowA.push(v);

          var diff = Math.abs(ref[i][j] - v);
          var m = diff > T ? 1 : 0;
          if (m) total++;
          rowM.push(m);
        }
        alin.push(rowA);
        mask.push(rowM);
      }

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var cr = document.createElement('div');
          cr.className = 'sim-ep0607_cell';
          cr.style.cssText = cellStyle(ref[i][j]);
          cr.textContent = ref[i][j];
          gRef.appendChild(cr);

          var ca = document.createElement('div');
          ca.className = 'sim-ep0607_cell';
          ca.style.cssText = cellStyle(alin[i][j]);
          ca.textContent = alin[i][j];
          gAlin.appendChild(ca);

          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.className = 'sim-ep0607_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'dx = ' + dx + '  |  T = ' + T + '  |  Total de pixels defeituosos: ' + total + ' / 9';
    }

    slDx.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep07(){
    var root = document.getElementById('sim-ep0607');
    if (root) initSim06Ep07(root); else setTimeout(tryInitSim06Ep07, 200);
  }
  tryInitSim06Ep07();
})();
</script>
""")

**Figure 6.7:** EP06_07 Simulator: *Pipeline* Inspection — Registration by Translation and Subtraction


<figure id="fig-06-sim-ep0607">
  <img src="imagens/fig-06-sim-ep0607.png" alt=" EP06_07 Simulator: *Pipeline* Inspection — Registration by Translation and Subtraction " style="max-width:80%" />
  <figcaption><strong>Figure 6.7:</strong>  EP06_07 Simulator: *Pipeline* Inspection — Registration by Translation and Subtraction </figcaption>
</figure>

In [ ]:
%%writefile EP06_07.py
# Python code

In [ ]:
TestSuite("EP06_07.py").run()

### EP06_08 ⚫ QRCode Segmentation and Real Decoding with OpenCV

In the previous exercises, the intermediate quantities of the image processing pipeline—such as areas, perimeters, variances, and displacements—were provided directly or calculated from numerical matrices, without the need for specialized Computer Vision libraries. In this chapter-closing exercise, this restriction is intentionally removed: the **OpenCV** library (`cv2`) will be used to locate and decode a real *QRCode* present in a scene.

The proposal reproduces a simplified workflow of systems employed in visual inspection, industrial automation, and automatic document reading. To keep the data input accessible to the educational context, image loading will be integrated into the didactic library `morph`, through the `mm.readImg` function.

The scene is provided in **ASCII PGM (P2)** format and contains a single valid *QRCode*, in addition to several **distractor objects**, such as rectangles, textured noise regions, and isolated blocks. Segmentation based solely on geometric properties—such as area and approximately square shape—is necessary to reduce the search space, but it is not sufficient to identify the correct code. The final confirmation will be performed exclusively by attempting to decode using `cv2.QRCodeDetector`, a procedure compatible with real automatic recognition applications.

#### 📋 Implementation Guidelines

1. **Reading dimensions and parameters**

   Read, in this order, from standard input:

   - one line containing the number of rows $L$;
   - one line containing the number of columns $C$;
   - one line containing the four algorithm parameters separated by spaces:
     - binarization threshold $T$ (integer);
     - minimum area $A_{\text{min}}$ (integer);
     - aspect tolerance $\text{tol}$ (real);
     - margin $M$ (integer, in pixels).

2. **Image loading**

   Use the didactic function `f = mm.readImg(L, C)` to read the $L \times C$ values of the grayscale image, obtaining a NumPy array of type `uint8`.

3. **Binarization**

   Apply inverted binary thresholding using the threshold $T$. Every pixel of the original image with intensity strictly greater than $T$ must be converted to 255, while the remaining ones must assume the value 0.

4. **Contour detection**

   Extract the external connected components using `cv2.findContours(...)` with the parameters:

   * `cv2.RETR_EXTERNAL`;
   * `cv2.CHAIN_APPROX_SIMPLE`.

5. **Geometric filtering**

   For each contour found:

   * compute the bounding rectangle `(x, y, w, h)` using `cv2.boundingRect`;
   * keep only candidates that simultaneously satisfy:

     **Minimum area**

     $$
     w \times h > A_{\text{min}}
     $$

     **Aspect ratio**

     $$
     \left|\frac{w}{h}-1\right| \le \text{tol}
     $$

6. **Candidate ordering**

   Sort the candidates by the bounding rectangle area

   $$
   w \times h
   $$

   in descending order.

   In case of a tie, preserve the order originally returned by `cv2.findContours`.

7. **Verification by decoding**

   For each candidate, following the established order:

   * expand the rectangle by $M$ pixels in all four directions;
   * clamp the indices to remain within the image;
   * extract the crop directly from the original image `f`;
   * apply `cv2.QRCodeDetector().detectAndDecode(...)` to that crop.

8. **Stopping criterion**

   Immediately stop processing when the first candidate yields a non-empty decoded string.

9. **Case not found**

   If no candidate is successfully decoded, print exactly: `QRCODE_NAO_ENCONTRADO`

10. **Output (found case)**

    Print two lines.

    First line: `linha coluna altura largura` using the **original** bounding rectangle, before the expansion by the margin $M$.

    Second line: `texto_decodificado`


#### 📌 Computational Restrictions

* Use OpenCV functions to perform binarization, contour detection, bounding rectangle computation, and QRCode decoding.
* Geometric filtering must necessarily occur before the decoding step.
* Use exclusively the fixed threshold $T$ provided in the input. It is not allowed to use automatic thresholding methods, such as Otsu or adaptive thresholding.
* Ensure that the crops sent to the decoder remain within the image boundaries.


#### 🧠 Theoretical Foundation

| Step                    | Role in the pipeline                                                                                                  | Consequence if omitted                                                                     |
| ------------------------ | ---------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------ |
| **Geometric filtering** | Reduces the search space by selecting only regions compatible with the expected QRCode geometry.                       | The decoder would process all contours, including noise and distractor objects.            |
| **Decoding**            | Semantically confirms whether the candidate contains a valid QRCode.                                                  | Geometrically similar objects could be incorrectly classified as a QRCode.                 |
| **Margin $M$**          | Preserves the quiet zone around the code, facilitating its detection.                                                | The absence of this margin may prevent proper alignment and correct reading of the code.   |

This exercise integrates concepts studied throughout the chapter into a single Computer Vision pipeline. Segmentation reduces the set of candidate regions through geometric characteristics, while the decoding step validates the content of the region using a specialized recognition algorithm.


#### 📦 Input and Output Specification (VPL)

**Input Structure**

```
L
C
T A_min tol M
[image matrix]
```

**Output Structure (Success)**

```
linha coluna altura largura
texto_decodificado
```

**Output Structure (Failure)**

```
QRCODE_NAO_ENCONTRADO
```

#### 📌 Reference Files (.pgm)

For validation purposes, local debugging, and analysis of real pixel matrices, the image files generated in ASCII P2 format are available in the project directory. You may use them to test the adherence of your code by decoding them with your cell phone (save the *.pgm files locally to view them):

* 📥 **[Case 1: Normal Pattern](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso1_Normal.pgm)** – Contains a single perfectly centered code with simple geometric distractors on the periphery.
* 📥 **[Case 2: Complex Scenario](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso2_Complexo.pgm)** – Presents a higher density of textured noise and multiple candidate distractors that test the limits of aspect filtering.
* 📥 **[Case 3: Expanded Message](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso3_MensagemSecreta.pgm)** – Contains a QRCode structured from a longer character string, generating a higher density of internal modules.
* 📥 **[Case 4: Compact Geometry](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso4_Excelente.pgm)** – Evaluates the pipeline behavior under optimized contrast conditions and borderline positioning.
* 📥 **[Case 5: Exclusion Scenario](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso5_Nao_Encontrado.pgm)** – An image composed purely of high-area distractor elements, designed to validate the controlled failure behavior of the program.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0608" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<!-- Cabeçalho no padrão institucional -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📋 Simulator EP06_08: QR Code Segmentation and Decoding</span>
  <span style="font-size:10px;font-weight:700;padding:3px 10px;border-radius:40px;border:1px solid #e4dcc8;background:#26241d;color:#7ee7c6;font-family:monospace;">Geometric Filter&rarr;Semantic Stopping</span>
</div>

<div style="padding:16px;background:#ffffff;">
  <p style="margin:0 0 14px 0;font-size:11px;color:#8a8371;line-height:1.5;text-align:center;font-weight:600;">
    Interactively adjust the algorithm's input parameters (A_min and tol) to check which components are geometrically filtered and how the semantic analysis stopping criterion interrupts the queue scan.
  </p>
  
  <div style="display:flex;gap:14px;margin-bottom:14px;flex-wrap:wrap;">
    <!-- Slider Area Minima -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Minimum area (A_min, px&sup2;)</label>
        <span id="sim-ep0608_vl_amin" style="font-family:monospace;font-weight:700;color:#26241d;">250</span>
      </div>
      <input id="sim-ep0608_sl_amin" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="3000" min="0" step="50" type="range" value="250">
    </div>
    
    <!-- Slider Tolerancia -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Aspect tolerance (tol)</label>
        <span id="sim-ep0608_vl_tol" style="font-family:monospace;font-weight:700;color:#26241d;">0.22</span>
      </div>
      <input id="sim-ep0608_sl_tol" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="1.0" min="0.05" step="0.01" type="range" value="0.22">
    </div>
  </div>

  <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:14px;margin-bottom:14px;">
    <!-- Canvas da Cena -->
    <div style="text-align:center;background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;letter-spacing:0.04em;">Scene Visualization (Matrix f)</div>
      <canvas id="sim-ep0608_canvas" width="260" height="260" style="border:1px solid #e4dcc8;border-radius:10px;background:#ffffff;margin:0 auto;display:block;"></canvas>
    </div>
    
    <!-- Lista de Candidatos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;text-align:center;letter-spacing:0.04em;">Connected Components in Queue</div>
      <div id="sim-ep0608_lista" style="font-family:monospace;font-size:11px;display:flex;flex-direction:column;gap:8px;"></div>
    </div>
  </div>
  
  <!-- Console de Saída VPL -->
  <div id="sim-ep0608_debug" style="background:#fafaf7;border-radius:12px;padding:12px;border:1px solid #e9e3d3;font-family:monospace;font-size:11px;color:#26241d;text-align:center;"></div>
</div>

<script>
(function(){
  function initSim06Ep08(root){
    if(!root || root.dataset.sim06Ep08Init) return;
    root.dataset.sim06Ep08Init = "1";

    var formas = [
      {x: 145, y: 35,  w: 76, h: 76, tipo: "Componente QRCode Real", cor: "#cbd5e1", decodifica: true, padrao: "qr"},
      {x: 35,  y: 145, w: 55, h: 68, tipo: "Falso QRCode (Assimétrico)", cor: "#e2e8f0", decodifica: false, padrao: "falso_qr"},
      {x: 45,  y: 35,  w: 44, h: 44, tipo: "Círculo / Distrator", cor: "#f1f5f9", decodifica: false, padrao: "circulo"},
      {x: 160, y: 175, w: 68, h: 26, tipo: "Retângulo Distrator", cor: "#e2e8f0", decodifica: false, padrao: "retangulo"},
      {x: 65,  y: 220, w: 14, h: 14, tipo: "Ruído Isolado", cor: "#f8fafc", decodifica: false, padrao: "ruido"}
    ];

    var slA = root.querySelector('#sim-ep0608_sl_amin');
    var vlA = root.querySelector('#sim-ep0608_vl_amin');
    var slT = root.querySelector('#sim-ep0608_sl_tol');
    var vlT = root.querySelector('#sim-ep0608_vl_tol');
    var canvas = root.querySelector('#sim-ep0608_canvas');
    var ctx = canvas.getContext('2d');
    var lista = root.querySelector('#sim-ep0608_lista');
    var dbg = root.querySelector('#sim-ep0608_debug');

    function desenhaForma(f, estado){
      ctx.save();
      
      var corBorda = '#94a3b8';
      if (estado === 'candidato_ok') corBorda = '#10b981';
      if (estado === 'candidato_falhou') corBorda = '#f43f5e';
      if (estado === 'rejeitado') corBorda = '#cbd5e1';

      ctx.lineWidth = (estado === 'candidato_ok' || estado === 'candidato_falhou') ? 3 : 1.5;
      ctx.strokeStyle = corBorda;

      if (estado === 'rejeitado') {
        ctx.fillStyle = '#f8fafc';
      } else {
        if(f.padrao === 'qr') ctx.fillStyle = '#e2e8f0';
        else if(f.padrao === 'retangulo') ctx.fillStyle = '#fffbeb';
        else if(f.padrao === 'falso_qr') ctx.fillStyle = '#f0f9ff';
        else ctx.fillStyle = '#fdf4ff';
      }

      if(f.padrao === 'circulo'){
        ctx.beginPath();
        ctx.arc(f.x + f.w/2, f.y + f.h/2, f.w/2, 0, 2 * Math.PI);
        ctx.fill(); ctx.stroke();
      } else {
        ctx.fillRect(f.x, f.y, f.w, f.h);
        ctx.strokeRect(f.x, f.y, f.w, f.h);
        
        if(f.padrao === 'qr' || f.padrao === 'falso_qr'){
          var c = f.w / 5;
          ctx.fillStyle = '#ffffff';
          [[f.x + 3, f.y + 3], [f.x + f.w - c - 3, f.y + 3], [f.x + 3, f.y + f.h - c - 3]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c, c);
            ctx.strokeRect(p[0], p[1], c, c);
          });
          
          ctx.fillStyle = (f.padrao === 'qr') ? '#334155' : '#64748b';
          [[f.x + 5, f.y + 5], [f.x + f.w - c + 1, f.y + 5], [f.x + 5, f.y + f.h - c + 1]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c - 4, c - 4);
          });
        }
      }
      ctx.restore();
    }

    function render(){
      var amin = parseInt(slA.value, 10);
      var tol = parseFloat(slT.value);
      vlA.textContent = amin;
      vlT.textContent = tol.toFixed(2);

      ctx.clearRect(0, 0, canvas.width, canvas.height);
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, canvas.width, canvas.height);

      var candidatos = formas.map(function(f){
        var area = f.w * f.h;
        var aspecto = f.w / f.h;
        var passaArea = area > amin;
        var passaAspecto = Math.abs(aspecto - 1.0) <= tol;
        return {f: f, area: area, aspecto: aspecto, passa: passaArea && passaAspecto};
      }).sort(function(a, b){ return b.area - a.area; });

      lista.innerHTML = '';
      var encontrado = null;
      var flagParada = false;

      candidatos.forEach(function(c){
        var estado, texto, bgBox, txBox;
        
        if(!c.passa){
          estado = 'rejeitado';
          texto = 'REJEITADO (Área = ' + c.area + ' px&sup2;, Aspeto = ' + c.aspecto.toFixed(2) + ')';
          bgBox = '#f1f5f9';
          txBox = '#94a3b8';
        } else if(flagParada){
          estado = 'rejeitado';
          texto = 'FILA INTERROMPIDA (Critério de Parada Ativo)';
          bgBox = '#f8fafc';
          txBox = '#cbd5e1';
        } else if(c.f.decodifica){
          estado = 'candidato_ok';
          texto = 'SUCESSO: DECODIFICADO &#10004;';
          bgBox = '#ecfdf5';
          txBox = '#059669';
          encontrado = c.f;
          flagParada = true;
        } else {
          estado = 'candidato_falhou';
          texto = 'GEOMETRIA OK &rarr; FALHA NA DECODIFICAÇÃO &#10008;';
          bgBox = '#fff5f5';
          txBox = '#e11d48';
        }
        
        desenhaForma(c.f, estado);
        
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 10px;border-radius:8px;background:' + bgBox + ';border:1px solid #edf2f7;color:' + txBox + ';display:flex;flex-direction:column;gap:2px;';
        
        var nameSpan = document.createElement('strong');
        nameSpan.style.fontSize = '11px';
        nameSpan.textContent = c.f.tipo + ' (' + c.area + ' px²)';
        
        var statusSpan = document.createElement('span');
        statusSpan.style.fontSize = '10px';
        statusSpan.style.opacity = '0.9';
        statusSpan.innerHTML = texto;

        div.appendChild(nameSpan);
        div.appendChild(statusSpan);
        lista.appendChild(div);
      });

      if (encontrado) {
        dbg.style.backgroundColor = '#eafaf1';
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.color = '#04342C';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#04342C;margin-bottom:4px;">&#128994; SAÍDA PADRÃO (VPL):</div>' +
                        'y=' + encontrado.y + ' x=' + encontrado.x + ' h=' + encontrado.h + ' w=' + encontrado.w + '<br>' +
                        '<span style="color:#04342C;font-weight:700;">"EP06_08 - PDI-VC | Parabens! Voce decodificou este QR Code!"</span>';
      } else {
        dbg.style.backgroundColor = '#fdecea';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.color = '#c0392b';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#c0392b;margin-bottom:4px;">&#128308; SAÍDA PADRÃO (VPL):</div>' +
                        'QRCODE_NAO_ENCONTRADO';
      }
    }

    slA.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  
  function tryInitSim06Ep08(){
    var root = document.getElementById('sim-ep0608');
    if(root) initSim06Ep08(root); else setTimeout(tryInitSim06Ep08, 200);
  }
  tryInitSim06Ep08();
})();
</script>
</div>
""")

**Figure 6.8:** Simulador EP06_08: Geometric Segmentation + Verification by QRCode Decoding


<figure id="fig-06-sim-ep0608">
  <img src="imagens/fig-06-sim-ep0608.png" alt=" Simulador EP06_08: Geometric Segmentation + Verification by QRCode Decoding " style="max-width:80%" />
  <figcaption><strong>Figure 6.8:</strong>  Simulador EP06_08: Geometric Segmentation + Verification by QRCode Decoding </figcaption>
</figure>

In [ ]:
%%writefile EP06_08.py
# Python Code

In [ ]:
TestSuite("EP06_08.py").run()